# Картографирование статистических данных


В этом разделе мы рассмотрим, как можно визуализировать данные на примере Базы данных муниципальных образований (БДМО) Росстата


## Импортируем библиотеки


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mapclassify as mc

## Картографирование БДМО Росстата


### Загружаем границы муниципалитетов

Тут небольшой пример, границы всех муниципальных образований верхнего уровня можно скачать [тут](https://geoportal.hse.ru/portal/apps/sites/#/geodata/datasets/12e78b9c3bf04feea944f9d53eb3d079/about)

Если нужно учитывать также изменения границ муниципальных образований со временем, потребуется отдельный набор данных (с историей границ).

> 📥 Файл с данными для этого примера (`muni2.geojson`) можно скачать [из этой папки на Google Drive](https://drive.google.com/drive/folders/12gsEj8kOjDP2ww36wgjW2xAXNljIIMfV?usp=sharing) и положить в папку `data`.

In [ ]:
muni = gpd.read_file('data/muni2.geojson')

muni.head()

### Загружаем и обрабатываем статистические данные


- выбираем/рассчитываем подходящие показатели
- проверяем полноту данных за разные годы


Данные БДМО от проекта "Если быть точным":

- Полный набор данных + документация [тут](https://tochno.st/datasets/bdmo)
- Данные по отдельным показателям [тут](https://docs.google.com/spreadsheets/d/1Z66eXdgzyBg2NPV4cRzrTtv6BSZFVaAi/edit?pli=1&gid=1495188554#gid=1495188554)

> 📥 Файлы с данными для этого примера (`bdmo_pop_sample.csv` и `bdmo_birth_sample.csv`) можно скачать [из этой папки на Google Drive](https://drive.google.com/drive/folders/12gsEj8kOjDP2ww36wgjW2xAXNljIIMfV?usp=sharing) и положить в папку `data`.

Читаем данные о населении


In [ ]:
population = pd.read_csv('data/bdmo_pop_sample.csv', sep=",")

population.head()

In [ ]:
population_filtered = population[population['mest'] == 'Все население']

population_wide = population_filtered.pivot_table(index=['oktmo', 'indicator', 'unit', 'region', 'munr', 'municipality'], columns='year', values='value').reset_index()

population_wide.head()

Смотрим на кол-во отсутствующих данных (чтобы понять, какие годы не стоит рассматривать)


In [ ]:
missing_values = population_wide.isna().sum()
missing_values 

Читаем данные о рождаемости


In [ ]:
birth = pd.read_csv('data/bdmo_birth_sample.csv', sep=",")

birth.head()


In [ ]:
birth_wide = birth.pivot_table(index=['oktmo', 'indicator', 'unit', 'region', 'munr', 'municipality'], columns='year', values='value').reset_index()

birth_wide.head()

In [ ]:
missing_values_birth = birth_wide.isna().sum()
missing_values_birth

### Объединяем данные на основе ОКТМО

**ОКТМО** — код территории по общероссийскому классификатору. Он есть и в геоданных (границы), и в статистике, поэтому именно по нему мы сшиваем таблицы: к каждому муниципалитету подтягиваем его показатели.


In [ ]:
# Привести оба поля к строковому типу
muni['oktmo'] = muni['oktmo'].astype(str)
population_wide['oktmo'] = population_wide['oktmo'].astype(str)
birth_wide['oktmo'] = birth_wide['oktmo'].astype(str)


muni_stat_pop = muni.merge(population_wide, 
                      left_on='oktmo',
                      right_on='oktmo', 
                      how='left',
                      suffixes=('', '_pop'))


muni_stat_all = muni_stat_pop.merge(birth_wide, 
                      left_on='oktmo',
                      right_on='oktmo', 
                      how='left',
                      suffixes=('', '_birth'))


muni_stat_all.head()


### Рассчитываем коэффициент естественного прироста в 2016 и 2020 годах


In [ ]:
muni_stat_all['natincrate2016'] = (muni_stat_all['2016_birth'] / muni_stat_all['2016']) * 1000
muni_stat_all['natincrate2020'] = (muni_stat_all['2020_birth'] / muni_stat_all['2020']) * 1000

Построим карту коэффициента естественного прироста за 2020 год:


In [ ]:
ax = muni_stat_all.plot(column='natincrate2020', cmap='BuPu', linewidth=0.01, edgecolor='0.75', legend=True, scheme="naturalBreaks")

# добавляем название
ax.set_title('Коэффициент естественного прироста, ‰')

# скрываем координатные оси
ax.axis('off')

plt.show()

Теперь сравним прирост за **2016 и 2020 годы**. Сначала построим две карты, где для каждого года интервалы классификации (`naturalBreaks`) считаются **отдельно**:


In [ ]:

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# 2016 год
plot1 = muni_stat_all.plot(
    column='natincrate2016',
    cmap='BuPu',
    linewidth=0.01, 
    edgecolor='0.75',
    legend=True,
    scheme="naturalBreaks",
    ax=ax1
)
ax1.set_title('Коэффициент естественного прироста, 2016, ‰')
ax1.axis('off')

# 2020 год
plot2 = muni_stat_all.plot(
    column='natincrate2020',
    cmap='BuPu',
    linewidth=0.01,
    edgecolor='0.75', 
    legend=True,
    scheme="naturalBreaks",
    ax=ax2
)
ax2.set_title('Коэффициент естественного прироста, 2020, ‰')
ax2.axis('off')

plt.tight_layout()
plt.show()

У карт выше есть подвох: интервалы (а значит, и легенда) у каждого года **свои**, поэтому напрямую сравнивать цвета между картами нельзя. Чтобы сравнение было корректным, зададим **единые интервалы** для обоих годов и построим карты заново:


In [ ]:
# Вычисляем общие интервалы на основе данных за оба года
all_values = np.concatenate([
    muni_stat_all['natincrate2016'].dropna().values,
    muni_stat_all['natincrate2020'].dropna().values
])

# Создаем классификатор с общими интервалами (k=5 -> 5 интервалов)
classifier = mc.NaturalBreaks(all_values, k=5)
common_bins = classifier.bins

print("Общие интервалы для обеих карт:", common_bins)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# 2016 год — с общими интервалами
muni_stat_all.plot(
    column='natincrate2016',
    cmap='BuPu',
    linewidth=0.01,
    edgecolor='0.75',
    legend=True,
    scheme="User_Defined",                       # пользовательские интервалы
    classification_kwds={'bins': common_bins},   # задаём общие интервалы
    ax=ax1
)
ax1.set_title('Коэффициент естественного прироста, 2016, ‰')
ax1.axis('off')

# 2020 год — те же самые интервалы
muni_stat_all.plot(
    column='natincrate2020',
    cmap='BuPu',
    linewidth=0.01,
    edgecolor='0.75',
    legend=True,
    scheme="User_Defined",
    classification_kwds={'bins': common_bins},
    ax=ax2
)
ax2.set_title('Коэффициент естественного прироста, 2020, ‰')
ax2.axis('off')

plt.tight_layout()
plt.show()

## Итог

На примере **БДМО Росстата** мы прошли полный путь картографирования статистических данных: загрузили границы муниципалитетов, подготовили и проверили статистику по годам, сшили таблицы с геометрией по **ОКТМО** и построили карты коэффициента естественного прироста.

Главное, что стоит запомнить: чтобы карты за разные годы можно было **сравнивать между собой**, интервалы классификации должны быть **общими** — иначе одинаковые цвета на разных картах будут означать разные значения.

Успехов!
